# Text Classification

## Representation Models

In [1]:
from datasets import load_dataset

# Load our data
data = load_dataset("rotten_tomatoes")
print(data)
data['train'][0, -1]


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})


{'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
  'things really get weird , though not particularly scary : the movie is all portent and no content .'],
 'label': [1, 0]}

### Task-Specific Model

A task-specific model is a representation model, such as BERT, trained for a specific task, like sentiment analysis. 

In [23]:
from transformers import pipeline
import torch 

# Path to our HF model 
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"
device = torch.device("mps" if torch.backends.mps.is_available() else 'cpu')

pipe = pipeline(
    model=model_path, 
    tokenizer=model_path, 
    return_all_scores=True,
    device=device
)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/Users/brianmorales/miniconda3/envs/thellmbook/lib/python3.10/site-packages/transformers/pipelines/text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead

In [50]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

print(KeyDataset(data, 'text').dataset)
print(KeyDataset(data, 'text').dataset['train']['text'][0])
print(KeyDataset(data, 'label').dataset['train']['label'][0])


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})
the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .
1


In [29]:
# Run inference
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
    negative_score = output[0]["score"]
    positive_score = output[2]["score"]
    assignment = np.argmax([negative_score, positive_score])
    y_pred.append(assignment)

100%|██████████| 1066/1066 [00:24<00:00, 43.58it/s]


In [30]:
from sklearn.metrics import classification_report

def evaluate_performance(y_true, y_pred): 
    # Create and print the classification report
    performance = classification_report(
        y_true, y_pred, 
        target_names=['Negative Review', 'Positive Review']
    )
    print(performance)

In [31]:
# Lets create our classification report
evaluate_performance(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.76      0.88      0.81       533
Positive Review       0.86      0.72      0.78       533

       accuracy                           0.80      1066
      macro avg       0.81      0.80      0.80      1066
   weighted avg       0.81      0.80      0.80      1066



Lets use a different model to get better performance

In [32]:
model_path="distilbert-base-uncased-finetuned-sst-2-english"
pipe = pipeline(
    model=model_path,
    tokenizer=model_path, 
    return_all_scores=True,
    device=device, 
)

/Users/brianmorales/miniconda3/envs/thellmbook/lib/python3.10/site-packages/transformers/pipelines/text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [33]:
y_pred = []
for output in tqdm(pipe(KeyDataset(data['test'], 'text')), total=len(data['test'])):
    neg_score = output[0]['score']
    pos_score = output[1]['score']
    assignment = np.argmax([neg_score, pos_score])
    y_pred.append(assignment)

100%|██████████| 1066/1066 [00:17<00:00, 60.16it/s]


In [34]:
evaluate_performance(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.89      0.90      0.90       533
Positive Review       0.90      0.89      0.90       533

       accuracy                           0.90      1066
      macro avg       0.90      0.90      0.90      1066
   weighted avg       0.90      0.90      0.90      1066



### Classification Leveraging Embeddings

In the previous example, we used a pretrained task-specific model for sentiment analysis. However, what if we cannot find a model that was pretrained for this specific task? Do we need to fine-tune a representation model ourselves? The answer is no!

#### Supervised Classification

We can perform part of the training process ourselves by approaching it from a more classical perspective. Instead of directly using the representation model for classification, we will use an embedding model for generating features. Those features can then be fed into a classifier, thereby creating a two-step approach as shown below 

![Alt text](figures/embedding-classification.png)

In [36]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# Convert text to embeddings
train_embeddings = model.encode(data['train']['text'], show_progress_bar=True)
test_embeddings = model.encode(data['test']['text'], show_progress_bar=True)

Batches:   0%|          | 0/267 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

In [52]:
train_embeddings.shape

(8530, 768)

Second step is to serve these embeddings as input features to our classifier - logistic regression model. 

In [54]:
from sklearn.linear_model import LogisticRegression

# Train a logistic regression on our train embeddings
clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data['train']['label'])

LogisticRegression(random_state=42)

In [55]:
# lets evaluate our model
y_pred = clf.predict(test_embeddings)
evaluate_performance(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.85      0.86      0.85       533
Positive Review       0.86      0.85      0.85       533

       accuracy                           0.85      1066
      macro avg       0.85      0.85      0.85      1066
   weighted avg       0.85      0.85      0.85      1066



### What if We Don't Have Labeled Data? 

We can perform zero-shot classification, where we have no labeled data to explore whether the task seems feasible. Zero-shot classification attempts to predict the labels of input text even though it was not trained on them. 

![Alt text](figures/zero-shot-learning.png)

We can describe our labels based on what they should represent. For example, a negative label for movie reviews can be described as “This is a negative movie review.” By describing and embedding the labels and documents, we have data that we can work with. 

In [56]:
label_embeddings = model.encode(["A negative review", "A positive review"])

To assign labels to documents, we can apply cosine similarity to the document label pairs. This is the cosine of the angle between vectors, which is calculated through the dot product of the embeddings and divided by the product of their lengths, as illustrated

![Alt text](figures/cosine-similarity.png)

We can use cosine similarity to check how similar a given document is to the description of the candidate labels. The label with the highest similarity to the document is chosen 

In [57]:
from sklearn.metrics.pairwise import cosine_similarity

# Find the best matching label for each document
sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis = 1)

In [58]:
# Lets see how well this method performs
evaluate_performance(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.78      0.77      0.78       533
Positive Review       0.77      0.79      0.78       533

       accuracy                           0.78      1066
      macro avg       0.78      0.78      0.78      1066
   weighted avg       0.78      0.78      0.78      1066



Lets try making the labels a bit more concrete and specific towards our data, "A very negative/positive movie review". This way the embedding will capture that it is a movie review and will focus a bit more on the extreme labels. 

In [59]:
extreme_label_embeddings = model.encode(['A very negative movie review', 'A very positive movie review'])
ex_sim_matrix = cosine_similarity(test_embeddings, extreme_label_embeddings)
y_pred = np.argmax(ex_sim_matrix, axis=1)
evaluate_performance(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.86      0.73      0.79       533
Positive Review       0.76      0.88      0.82       533

       accuracy                           0.80      1066
      macro avg       0.81      0.80      0.80      1066
   weighted avg       0.81      0.80      0.80      1066



## Generative Models

Classification with generative language models, such as OpenAI’s GPT models, works a bit differently from what we have done thus far. These models take as input some text and generative text and are thereby aptly named sequence-to-sequence models. These generative models are generally trained on a wide variety of tasks and usually do not perform your use case out of the box. Instead, we need to help it understand the context and guide it toward the answers that we are looking for. This guiding process is done mainly through the instruction, or prompt, that you give such a model. Iteratively improving your prompt to get your preferred output is called prompt engineering, as shown below

![Alt text](figures/prompt-engineering.png)

### Text-to-Text Transfer Transformer

In [61]:
# We use the pretrained Flan-T5 model for classification
pipe = pipeline(
    task="text2text-generation", 
    model="google/flan-t5-small", 
    device=device
)

Compared to our task-specific model, we cannot just give the model some text and hope it will output the sentiment. Instead, we will have to instruct the model to do so.

In [63]:
# Prepare our data
prompt = "Is the following sentence positive or negative? "
data = data.map(lambda example: {"t5": prompt + example['text']})
data

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
})

In [64]:
# run inference
y_pred = []
for output in tqdm(pipe(KeyDataset(data['test'], 't5')), total=len(data['test'])):
    text = output[0]["generated_text"]
    y_pred.append(0 if text == 'negative' else 1)

  0%|          | 0/1066 [00:00<?, ?it/s]/Users/brianmorales/miniconda3/envs/thellmbook/lib/python3.10/site-packages/transformers/generation/utils.py:1168: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
100%|██████████| 1066/1066 [00:55<00:00, 19.22it/s]


In [65]:
evaluate_performance(data['test']['label'], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.83      0.85      0.84       533
Positive Review       0.85      0.83      0.84       533

       accuracy                           0.84      1066
      macro avg       0.84      0.84      0.84      1066
   weighted avg       0.84      0.84      0.84      1066



### ChatGPT for Classification

Although the underlying architecture of the original ChatGPT model (GPT-3.5) is not shared, we can assume from its name that it is based on the decoder-only architecture that we have seen in the GPT models thus far. 

Fortunately, OpenAI shared an [overview of the training procedure](https://openai.com/index/chatgpt/) that involved an important component, namely preference tuning. OpenAI first manually created the desired output to an input prompt (instruction data) and used that data to create a first variant of its model.

Then OpenAI used the resulting model to generate multiple outputs that were manually ranked from best to worst. This ranking demonstrates a preference for certain outputs (preference data) and was used to create its final model, ChatGPT. A major benefit of using preference data over instruction data is the nuance it represents. By demonstrating the difference between a good and better output the generative model learns to generate text that resembles human preference.

Using the client, we create the `chatgpt_generation` function, which allows us to generate some text based on a specific prompt, input document, and the selected model.

In [67]:
import openai

# Create client
client = openai.OpenAI(api_key="")

def chatgpt_generation(prompt, document, model="gpt-3.5-turbo-0125"):
    # Generate an output based on a prompt and an input document 
    messages = [
        {
            'role': 'system', 
            'content': 'You are a helpful assistant.'
        }, 
        {
            'role': 'user',
            'content': prompt.replace('[DOCUMENT]', document)
        }
    ]
    chat_completion = client.chat.completions.create(
        messages=messages, 
        model=model, 
        temperature=0
    )
    return chat_completion.choices[0].message.content

In [68]:
# Next we will need to create template to ask the model to perform the classification
prompt = """ Predict whether the following document is a positive or negative movie review:

[DOCUMENT]

If it is positive return 1 and if it is negative return 0. Do not give any other answers.
"""

# Predict the target using GPT
document = "unpretentious, charming, quirky, original"
chatgpt_generation(prompt, document)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [69]:
# You can skip this if you want to save your (free) credits
predictions = [
    chatgpt_generation(prompt, doc) for doc in tqdm(data["test"]["text"])
]

  0%|          | 0/1066 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:
# Extract predictions
y_pred = [int(pred) for pred in predictions]

# Evaluate performance
evaluate_performance(data["test"]["label"], y_pred)

The F1 score of 0.91 already gives a glimpse into the performance of the model that brought generative AI to the masses. However, since we do not know what data the model was trained on, we cannot easily use these kinds of metrics for evaluating the model. For all we know, it might have actually been trained on our dataset!

# Conclusion

We explored text classification using both generative and representation language models. Our goal was to assign a label or class to input text for the classification of a review’s sentiment.

We explored two types of representation models, a task-specific model and an embedding model. The task-specific model was pretrained on a large dataset specifically for sentiment analysis and showed us that pretrained models are a great technique for classifying documents. The embedding model was used to generate multipurpose embeddings that we used as the input to train a classifier.

Similarly, we explored two types of generative models, an open source encoder-decoder model (Flan-T5) and a closed source decoder-only model (GPT-3.5). We used these generative models in text classification without requiring specific (additional) training on domain data or labeled datasets.